# Track labels in 2-channel time-lapse stacks

For each 2-channel time-lapse (`ch0 = brightfield`, `ch1 = binary mask`) under `stack_outputs`:

1. Extract the mask channel and fill holes, then label objects per frame.
2. Link objects over time with TrackMate's Advanced Kalman tracker (no splitting/merging).
3. Save a 2-channel output (`ch0 = brightfield`, `ch1 = linked track labels`) as a 32-bit ImageJ TIFF with the pixel calibration embedded. Tracked objects keep a consistent id across frames; objects that were never linked to a track are given the value **-1**.
4. Write one morphology CSV per subfolder: every object id, its source image, and morphological parameters (area in µm² and px², eccentricity, solidity, ...) at each timepoint.


In [1]:
import logging
import os
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
import imagej
from imagej import Mode
import scyjava as sj
from scipy import ndimage as ndi
from skimage.measure import label as cc_label, regionprops_table

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
LOGGER = logging.getLogger("tracking")


In [2]:
# --- Configuration -----------------------------------------------------------
INPUT_ROOT = Path(r"C:\Users\taylorhearn\The University of Manchester Dropbox\Isobel Taylor-Hearn\marta_labels\stack_outputs")
OUTPUT_ROOT = INPUT_ROOT.parent / "stack_outputs_tracked"

# Subfolders under INPUT_ROOT to skip (e.g. the archived "old" copies).
SKIP_SUBFOLDERS = {"old"}

# Channel roles inside each input stack (T, C, Y, X).
BRIGHTFIELD_CHANNEL = 0
MASK_CHANNEL = -1  # last channel

# Pixel calibration. Value given as cm; everything else is derived from it.
PIXEL_SIZE_CM = 0.0064500
PIXEL_SIZE_UM = PIXEL_SIZE_CM * 10_000.0  # 1 cm = 10,000 µm
PIXEL_AREA_UM2 = PIXEL_SIZE_UM ** 2

# TrackMate Advanced Kalman tracker settings.
INITIAL_SEARCH_RADIUS = 200.0   # frame-to-frame linking distance (LINKING_MAX_DISTANCE)
SEARCH_RADIUS = 400.0          # Kalman search radius (KALMAN_SEARCH_RADIUS)
MAX_FRAME_GAP = 2
QUALITY_PENALTY = 0.2          # feature penalty applied to QUALITY
ALLOW_TRACK_SPLITTING = False
ALLOW_TRACK_MERGING = False

print("Output root:", OUTPUT_ROOT)
print(f"Pixel size: {PIXEL_SIZE_CM} cm = {PIXEL_SIZE_UM} µm/px  ->  {PIXEL_AREA_UM2:.4f} µm²/px")


Output root: C:\Users\taylorhearn\The University of Manchester Dropbox\Isobel Taylor-Hearn\marta_labels\stack_outputs_tracked
Pixel size: 0.00645 cm = 64.5 µm/px  ->  4160.2500 µm²/px


In [3]:
# --- Mask preparation and morphology -----------------------------------------

def prepare_instance_labels(mask_stack):
    """Fill holes in the binary mask and label objects per frame.

    Parameters
    ----------
    mask_stack : np.ndarray
        (T, Y, X) mask channel. Any non-zero pixel is treated as foreground.

    Returns
    -------
    np.ndarray
        (T, Y, X) uint16 stack where each object in each frame has a unique
        per-frame integer label (background = 0).
    """
    num_frames = mask_stack.shape[0]
    labelled = np.zeros(mask_stack.shape, dtype=np.uint16)
    for frame_index in range(num_frames):
        foreground = mask_stack[frame_index] > 0
        foreground = ndi.binary_fill_holes(foreground)
        labelled[frame_index] = cc_label(foreground).astype(np.uint16)
    return labelled


def measure_morphology(linked_labels, source_name, pixel_size_um):
    """Compute per-object, per-timepoint morphology from a tracked label stack.

    Parameters
    ----------
    linked_labels : np.ndarray
        (T, Y, X) stack where every object carries the same id across frames.
        Unlinked objects (value -1) are ignored.
    source_name : str
        Name of the source image (stored in the ``image`` column).
    pixel_size_um : float
        Physical pixel size in micrometres.

    Returns
    -------
    pandas.DataFrame
        One row per (object_id, timepoint) with morphological parameters.
    """
    properties = ("label", "area", "perimeter", "eccentricity", "solidity", "extent",
                  "major_axis_length", "minor_axis_length", "orientation",
                  "equivalent_diameter_area", "centroid")
    per_frame_tables = []
    for frame_index in range(linked_labels.shape[0]):
        frame_labels = np.clip(linked_labels[frame_index], 0, None)  # drop -1 unlinked objects
        if frame_labels.max() == 0:
            continue
        table = pd.DataFrame(regionprops_table(frame_labels, properties=properties))
        table.insert(0, "timepoint", frame_index)
        per_frame_tables.append(table)

    if not per_frame_tables:
        return pd.DataFrame()

    morphology = pd.concat(per_frame_tables, ignore_index=True)
    morphology = morphology.rename(columns={
        "label": "object_id",
        "area": "area_pixels",
        "perimeter": "perimeter_pixels",
        "major_axis_length": "major_axis_length_pixels",
        "minor_axis_length": "minor_axis_length_pixels",
        "equivalent_diameter_area": "equivalent_diameter_pixels",
        "centroid-0": "centroid_y",
        "centroid-1": "centroid_x",
    })
    morphology["area_um2"] = morphology["area_pixels"] * (pixel_size_um ** 2)
    morphology["perimeter_um"] = morphology["perimeter_pixels"] * pixel_size_um
    morphology["major_axis_length_um"] = morphology["major_axis_length_pixels"] * pixel_size_um
    morphology["minor_axis_length_um"] = morphology["minor_axis_length_pixels"] * pixel_size_um
    morphology["equivalent_diameter_um"] = morphology["equivalent_diameter_pixels"] * pixel_size_um
    morphology["aspect_ratio"] = morphology["major_axis_length_pixels"] / morphology["minor_axis_length_pixels"].replace(0, np.nan)
    morphology.insert(0, "image", source_name)

    ordered = ["object_id", "image", "timepoint",
               "area_pixels", "area_um2",
               "perimeter_pixels", "perimeter_um",
               "eccentricity", "solidity", "extent", "aspect_ratio",
               "major_axis_length_pixels", "major_axis_length_um",
               "minor_axis_length_pixels", "minor_axis_length_um",
               "equivalent_diameter_pixels", "equivalent_diameter_um",
               "orientation", "centroid_y", "centroid_x"]
    return morphology[ordered].sort_values(["object_id", "timepoint"]).reset_index(drop=True)


In [4]:
# --- TrackMate linking -------------------------------------------------------

def run_trackmate_linking(label_stack_path):
    """Link labelled objects over time with TrackMate's Advanced Kalman tracker.

    Parameters
    ----------
    label_stack_path : str or Path
        Path to a single-channel (T, Y, X) instance-label TIFF.

    Returns
    -------
    dict
        Keys: ``tracks_df`` (per-spot dataframe), ``model``, ``settings``,
        ``image_plus`` (the TrackMate ImagePlus, for XML export).
    """
    IJ = sj.jimport("ij.IJ")
    HashMap = sj.jimport("java.util.HashMap")
    Integer = sj.jimport("java.lang.Integer")
    Double = sj.jimport("java.lang.Double")
    Model = sj.jimport("fiji.plugin.trackmate.Model")
    Settings = sj.jimport("fiji.plugin.trackmate.Settings")
    TrackMate = sj.jimport("fiji.plugin.trackmate.TrackMate")
    Logger = sj.jimport("fiji.plugin.trackmate.Logger")
    LabelImageDetectorFactory = sj.jimport("fiji.plugin.trackmate.detection.LabelImageDetectorFactory")
    AdvancedKalmanTrackerFactory = sj.jimport("fiji.plugin.trackmate.tracking.kalman.AdvancedKalmanTrackerFactory")

    image_plus = IJ.openImage(str(label_stack_path))
    if image_plus is None:
        raise RuntimeError(f"Could not open label stack: {label_stack_path}")
    num_timepoints = int(image_plus.getNFrames())
    if num_timepoints <= 1:
        num_timepoints = int(image_plus.getStackSize())
    image_plus.setDimensions(1, 1, num_timepoints)
    image_plus.setOpenAsHyperStack(True)

    model = Model()
    model.setLogger(Logger.VOID_LOGGER)
    settings = Settings(image_plus)
    settings.detectorFactory = LabelImageDetectorFactory()
    detector_settings = HashMap()
    detector_settings.put("TARGET_CHANNEL", Integer.valueOf(1))
    detector_settings.put("SIMPLIFY_CONTOURS", False)
    settings.detectorSettings = detector_settings

    settings.trackerFactory = AdvancedKalmanTrackerFactory()
    tracker_settings = HashMap(settings.trackerFactory.getDefaultSettings())
    tracker_settings.put("KALMAN_SEARCH_RADIUS", Double.valueOf(float(SEARCH_RADIUS)))
    tracker_settings.put("LINKING_MAX_DISTANCE", Double.valueOf(float(INITIAL_SEARCH_RADIUS)))
    tracker_settings.put("MAX_FRAME_GAP", Integer.valueOf(int(MAX_FRAME_GAP)))
    tracker_settings.put("ALLOW_TRACK_SPLITTING", bool(ALLOW_TRACK_SPLITTING))
    tracker_settings.put("ALLOW_TRACK_MERGING", bool(ALLOW_TRACK_MERGING))
    quality_penalty = HashMap()
    quality_penalty.put("QUALITY", Double.valueOf(float(QUALITY_PENALTY)))
    tracker_settings.put("LINKING_FEATURE_PENALTIES", quality_penalty)
    tracker_settings.put("GAP_CLOSING_FEATURE_PENALTIES", quality_penalty)
    settings.trackerSettings = tracker_settings
    settings.addAllAnalyzers()

    trackmate = TrackMate(model, settings)
    if not trackmate.checkInput():
        raise RuntimeError(f"TrackMate input rejected: {trackmate.getErrorMessage()}")
    if not trackmate.process():
        raise RuntimeError(f"TrackMate processing failed: {trackmate.getErrorMessage()}")

    track_model = model.getTrackModel()
    spot_records = []
    for track_id in track_model.trackIDs(True):
        for spot in track_model.trackSpots(track_id):
            spot_records.append({
                "track_id": int(track_id),
                "t": int(float(spot.getFeature("FRAME"))),
                "y": float(spot.getFeature("POSITION_Y")),
                "x": float(spot.getFeature("POSITION_X")),
                "quality": float(spot.getFeature("QUALITY")),
            })
    tracks_df = pd.DataFrame(spot_records, columns=["track_id", "t", "y", "x", "quality"])
    return {"tracks_df": tracks_df, "model": model, "settings": settings, "image_plus": image_plus}


def paint_linked_labels(instance_labels, tracks_df):
    """Re-paint per-frame instance labels with globally consistent track ids.

    Each tracked object is looked up by its spot centroid and every pixel of
    that object in that frame is set to ``track_id + 1`` (label 0 = background).
    Foreground objects that were never linked to a track are set to -1.
    """
    linked = np.zeros(instance_labels.shape, dtype=np.int32)
    num_frames, num_y, num_x = instance_labels.shape
    for track_id, frame, y, x in tracks_df[["track_id", "t", "y", "x"]].to_numpy():
        frame_index = int(frame)
        if not 0 <= frame_index < num_frames:
            continue
        y_index = int(np.clip(round(float(y)), 0, num_y - 1))
        x_index = int(np.clip(round(float(x)), 0, num_x - 1))
        source_label = int(instance_labels[frame_index, y_index, x_index])
        if source_label > 0:
            linked[frame_index, instance_labels[frame_index] == source_label] = int(track_id) + 1
    linked[(instance_labels > 0) & (linked == 0)] = -1
    return linked


In [5]:
# --- Per-image processing ----------------------------------------------------

def save_tracked_imagej_tiff(output_path, brightfield, linked_labels):
    """Save a 2-channel (brightfield, linked labels) ImageJ TIFF with calibration.

    Written as 32-bit float so unlinked objects can carry the value -1.
    """
    output_stack = np.stack([brightfield.astype(np.float32), linked_labels.astype(np.float32)], axis=1)  # (T, C, Y, X)
    resolution = (1.0 / PIXEL_SIZE_CM, 1.0 / PIXEL_SIZE_CM)  # pixels per cm
    tifffile.imwrite(
        str(output_path),
        output_stack,
        imagej=True,
        resolution=resolution,
        metadata={"axes": "TCYX", "unit": "cm", "mode": "composite"},
    )


def process_single_image(image_path, output_dir):
    """Process one 2-channel stack: track, save outputs, return morphology rows."""
    output_dir.mkdir(parents=True, exist_ok=True)
    stem = image_path.stem

    stack = tifffile.imread(str(image_path))  # (T, C, Y, X)
    brightfield = stack[:, BRIGHTFIELD_CHANNEL]
    mask = stack[:, MASK_CHANNEL]

    instance_labels = prepare_instance_labels(mask)

    temp_handle, temp_name = tempfile.mkstemp(suffix=".tif")
    os.close(temp_handle)
    try:
        tifffile.imwrite(temp_name, instance_labels)
        result = run_trackmate_linking(temp_name)
    finally:
        try:
            os.remove(temp_name)
        except OSError:
            pass

    linked_labels = paint_linked_labels(instance_labels, result["tracks_df"])

    output_image_path = output_dir / f"{stem}_tracked.tif"
    save_tracked_imagej_tiff(output_image_path, brightfield, linked_labels)

    morphology = measure_morphology(linked_labels, stem, PIXEL_SIZE_UM)
    n_tracks = int(result["tracks_df"]["track_id"].nunique())
    LOGGER.info("%s: %d tracks -> %s", stem, n_tracks, output_image_path.name)
    return morphology


In [6]:
# --- Initialise ImageJ / TrackMate (run once) --------------------------------
ij = imagej.init("sc.fiji:fiji", mode=Mode.HEADLESS, add_legacy=True)
print("ImageJ version:", ij.getVersion())


ImageJ version: 2.17.0/1.54p


In [7]:
# --- Batch process every subfolder ------------------------------------------
subfolders = sorted(
    p for p in INPUT_ROOT.iterdir()
    if p.is_dir() and p.name not in SKIP_SUBFOLDERS
)
print("Subfolders to process:", [p.name for p in subfolders])

for subfolder in subfolders:
    output_subdir = OUTPUT_ROOT / subfolder.name
    image_paths = sorted(subfolder.glob("*.tif"))
    morphology_frames = []

    for image_path in image_paths:
        try:
            morphology = process_single_image(image_path, output_subdir)
            if not morphology.empty:
                morphology_frames.append(morphology)
        except Exception as error:
            LOGGER.error("Failed on %s: %s", image_path.name, error)

    if morphology_frames:
        subfolder_morphology = pd.concat(morphology_frames, ignore_index=True)
        csv_path = output_subdir / f"{subfolder.name}_morphology.csv"
        output_subdir.mkdir(parents=True, exist_ok=True)
        subfolder_morphology.to_csv(csv_path, index=False)
        LOGGER.info("%s: wrote %d rows -> %s", subfolder.name, len(subfolder_morphology), csv_path)
    else:
        LOGGER.warning("%s: no morphology rows produced.", subfolder.name)

print("Done.")


Subfolders to process: ['ADC15T', 'LCNEC11', 'LCNEC23', 'LCNEC26', 'LCNEC27', 'LNET10d']


2026-09-08 11:42:30,900 INFO r01c03f17p01-ch1sk1fk1fl1_stack_15T_labels: 9 tracks -> r01c03f17p01-ch1sk1fk1fl1_stack_15T_labels_tracked.tif
2026-09-08 11:42:36,044 INFO r01c03f34p01-ch1sk1fk1fl1_stack_15T_labels: 3 tracks -> r01c03f34p01-ch1sk1fk1fl1_stack_15T_labels_tracked.tif
2026-09-08 11:42:41,166 INFO r02c03f22p01-ch1sk1fk1fl1_stack_15T_labels: 3 tracks -> r02c03f22p01-ch1sk1fk1fl1_stack_15T_labels_tracked.tif
2026-09-08 11:42:46,211 INFO r02c03f29p01-ch1sk1fk1fl1_stack_15T_labels: 1 tracks -> r02c03f29p01-ch1sk1fk1fl1_stack_15T_labels_tracked.tif
2026-09-08 11:42:50,150 INFO r03c03f02p01-ch1sk1fk1fl1_stack_15T_labels: 1 tracks -> r03c03f02p01-ch1sk1fk1fl1_stack_15T_labels_tracked.tif
2026-09-08 11:42:55,585 INFO r03c03f10p01-ch1sk1fk1fl1_stack_15T_labels: 4 tracks -> r03c03f10p01-ch1sk1fk1fl1_stack_15T_labels_tracked.tif
2026-09-08 11:42:59,924 INFO r03c03f28p01-ch1sk1fk1fl1_stack_15T_labels: 1 tracks -> r03c03f28p01-ch1sk1fk1fl1_stack_15T_labels_tracked.tif
2026-09-08 11:43:04,

Done.
